# Video Scene Analysis — Full Object & Event Detection Demo - Enhanced Causal Reasoning Demo

This notebook demonstrates the end-to-end pipeline for analysing dashcam / CCTV
footage using **Gemini 2.5 Pro** via **LiteLLM**. Unlike the crash-only variant,
this version instructs the model to **describe every visible object, road user,
environmental feature, and event** in each frame — not just collisions.

**Enhancement**
* **Causal reasoning prompts** that force the model to trace why vehicles stop rather than assuming they were "inexplicably stationary".
* **Key-frame extraction** at critical timestamps — still images are sent alongside the video to give the model higher spatial resolution at the moments that matter most.
* **Two-pass verification** — a second LLM call cross-checks the initial analysis, specifically targeting temporal attribution errors (e.g. misidentifying which vehicle caused a collision).

**Pipeline steps:**
1. Configure LiteLLM connection
2. Define extended Pydantic schemas for full scene description
3. Load & validate a sample dashcam video
4. Extract key frames at critical timestamps
5. Encode the video to base64 and build the multimodal payload
6. Craft the enhanced prompt & Send the request to Gemini 2.5 Pro for scene-level analysis
7. Run verification pass (Pass 2)
8. Parse the structured JSON response into Pydantic models
9. Display and explore the results

## 1. Setup & Configuration

In [ ]:
import litellm
import json
import base64
import logging
import cv2
import sys
import numpy as np
from collections.abc import Sequence
from pathlib import Path
from IPython.display import display, JSON, Video, Image as IPImage

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# ── LiteLLM credentials ─────────────────────────────────────────────
# Replace with your own credentials or load from environment variables.

LITELLM_API_KEY = ""
LITELLM_BASE_URL = ""

litellm.api_key = LITELLM_API_KEY
litellm.api_base = LITELLM_BASE_URL
litellm.suppress_debug_info = True

# Model identifier — routed through LiteLLM to the Gemini 2.5 Pro deployment
MODEL_NAME = "gemini/gemini-2.5-pro"

print(f"LiteLLM configured — base: {LITELLM_BASE_URL}")
print(f"Model: {MODEL_NAME}")

## 2. Import Pydantic Schemas for Scene Description

These schemas are defined in `video_analysis_poc.video_llm.schema.video`.

In [ ]:
# Ensure the project root is on sys.path for local imports
sys.path.insert(0, str(Path(".").resolve().parent))


from video_analysis_poc.video_llm.schema.video import (
    DetectedObject,
    CrashInvolvedParty,
    WitnessedCrashDetail,
    SceneEvent,
    EnvironmentContext,
    VideoSceneAnalysisOutput,
)
print("Imported scene-description schemas from video_analysis_poc.video_llm.schema.video")

## 3. Load & Validate the Sample Video

In [ ]:
# ── Point this to your local dashcam / CCTV sample ─────────────────
VIDEO_PATH = Path("C:\\Users\\AMIR.RAHIMI\\Downloads\\Dashcam\\Inbound_CHO_Dashcam_20-50-780086.mp4")

SUPPORTED_FORMATS = {"mp4", "avi", "mov", "mkv", "webm"}
MAX_VIDEO_SIZE_MB = 100.0

VIDEO_MIME_TYPES = {
    "mp4": "video/mp4",
    "avi": "video/x-msvideo",
    "mov": "video/quicktime",
    "mkv": "video/x-matroska",
    "webm": "video/webm",
}

In [ ]:
def validate_video(path: Path) -> None:
    """Validate that the video exists, has a supported format, and is within size limits."""
    if not path.exists():
        raise FileNotFoundError(f"Video file not found: {path}")

    ext = path.suffix.lstrip(".").lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f"Unsupported format '.{ext}'. Supported: {SUPPORTED_FORMATS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_VIDEO_SIZE_MB:
        raise ValueError(f"Video is {size_mb:.1f} MB, exceeding limit of {MAX_VIDEO_SIZE_MB} MB.")

    print(f"Video validated: {path.name}  ({size_mb:.2f} MB, format: .{ext})")


validate_video(VIDEO_PATH)

In [ ]:
# ── Preview the video inline (works for .mp4 in JupyterLab) ────────
try:
    display(Video(str(VIDEO_PATH), embed=True, width=640))
except Exception as e:
    print(f"Inline preview not available: {e}")
    print(f"Open the file manually: {VIDEO_PATH.resolve()}")

## 4. Extract Key Frames at Critical Timestamps (e.g., around the crash event)

Supplementing the full video with still images at critical moments gives the model better spatial resolution exactly when it matters — particularly around the moments when vehicles interact closely (e.g. a truck and car entering a roundabout side-by-side).

In [ ]:


def extract_key_frames(
    video_path: Path,
    timestamps: Sequence[float],
    max_dim: int = 768,
) -> list[tuple[float, str]]:
    """Extract and base64-encode frames at specific timestamps.

    Parameters:
        video_path: Path to the video file.
        timestamps: List of timestamps (in seconds) to extract.
        max_dim: Maximum dimension (width or height) for resized output.

    Returns:
        List of (timestamp, base64_jpeg_string) tuples.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames: list[tuple[float, str]] = []

    for ts in timestamps:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(ts * fps))
        ret, frame = cap.read()
        if not ret:
            logger.warning("Could not read frame at %.1f s", ts)
            continue

        h, w = frame.shape[:2]
        scale = min(max_dim / w, max_dim / h, 1.0)
        if scale < 1.0:
            frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

        _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
        frames.append((ts, b64))

    cap.release()
    return frames


# ── Define critical timestamps ─────────────────────────────────────
# Approach & context
APPROACH_TS = [0, 5, 10, 14]
# Roundabout entry — truck + car interaction zone (highest priority)
ENTRY_TS = [15, 16, 17, 18, 19, 20, 21, 22]
# Post-incident: stationary vehicle, later traffic navigating around
POST_TS = [25, 30, 35, 38, 39, 40, 42, 45, 50]

KEY_TIMESTAMPS = APPROACH_TS + ENTRY_TS + POST_TS

logger.info("Extracting %d key frames …", len(KEY_TIMESTAMPS))
key_frames = extract_key_frames(VIDEO_PATH, KEY_TIMESTAMPS)
print(f"Extracted {len(key_frames)} key frames.")
print(f"Timestamps: {[t for t, _ in key_frames]}")

In [ ]:
# ── Quick visual sanity check — show a few key frames ─────────────
import io
from PIL import Image as PILImage

_preview_ts = [15, 18, 20, 25, 39, 42]
for ts, b64 in key_frames:
    if ts in _preview_ts:
        img_bytes = base64.b64decode(b64)
        display(IPImage(data=img_bytes, format="jpeg", width=480))
        print(f"  ↑ Frame at {ts:.0f} s")

## 4. Encode Video & Build Multimodal Payload

In [ ]:
def encode_video_to_base64(path: Path) -> str:
    """Read a video file and return its base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_mime_type(path: Path) -> str:
    ext = path.suffix.lstrip(".").lower()
    return VIDEO_MIME_TYPES.get(ext, "video/mp4")


logger.info("Encoding video to base64 …")
video_b64 = encode_video_to_base64(VIDEO_PATH)
video_mime = get_mime_type(VIDEO_PATH)

print(f"Encoded: {len(video_b64) / (1024*1024):.2f} MB base64 string  (MIME: {video_mime})")

In [ ]:
# ── Build the multimodal content parts ──────────────────────────────
CAMERA_TYPE = "dashcam_front"

video_content_parts = [
    {
        "type": "text",
        "text": f"[{CAMERA_TYPE}] Video file: {VIDEO_PATH.name}",
    },
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{video_mime};base64,{video_b64}",
        },
    },
]

# ── Append key frames as supplementary still images ─────────────────
# These give the model pixel-level detail at the moments that matter.
for ts, b64 in key_frames:
    video_content_parts.append({
        "type": "text",
        "text": (
            f"[STILL FRAME at {ts:.0f}s] — Examine this frame carefully. "
            f"Pay close attention to the spatial relationship between ALL "
            f"vehicles visible, especially large vehicles and adjacent cars."
        ),
    })
    video_content_parts.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
    })

print(f"Built {len(video_content_parts)} content parts "
      f"(1 video + {len(key_frames)} annotated key frames).")

## 5. Craft the Enhanced Prompt & Send to Gemini 2.5 Pro (Pass 1)

The system and task prompts have been rewritten to:
* Require **causal reasoning** — the model must explain why vehicles stop.
* Highlight **truck swept-path collisions** as a common pattern.
* Warn against **temporal attribution errors** (blaming a later-arriving vehicle for a collision that already occurred).
* Explicitly consider **hit-and-run / scrape-and-leave** scenarios.


In [ ]:
# ── System prompt — general scene understanding ────────────────────
SYSTEM_PROMPT = (
    "You are an expert multimodal video analyst specialising in road scene "
    "understanding from dashcam and CCTV footage. Your task is to provide a "
    "comprehensive, structured description of EVERYTHING visible in the video: "
    "all vehicles, pedestrians, cyclists, road infrastructure, signage, lane "
    "markings, buildings, vegetation, weather and lighting conditions, and any "
    "notable events (lane changes, braking, turns, pedestrian crossings, etc.). "
    "Pay special attention to any collisions or near-miss events — whether the "
    "dashcam vehicle is directly involved OR merely witnesses a crash between "
    "other road users. For every collision or near-miss, provide a detailed "
    "forensic-level breakdown: impact type, each involved party's description "
    "and actions before/after, probable cause, fault indicators, sequence of "
    "events, post-crash actions, and estimated speeds. Be thorough and precise "
    "with timestamps. Return your findings as a structured JSON object.\n\n"
    "IMPORTANT ANALYSIS PRINCIPLES:\n"
    "- Always establish CAUSAL relationships between events. If a vehicle is "
    "stopped in an unusual location, determine what caused it to stop by "
    "examining the preceding seconds — do NOT assume it was inexplicably "
    "stationary without a reason.\n"
    "- Large vehicles like trucks and HGVs have wide turning arcs that "
    "frequently cause sideswipe collisions with adjacent vehicles on "
    "roundabouts and tight curves. Look carefully for this pattern whenever "
    "a truck and a smaller vehicle enter a roundabout side by side.\n"
    "- The striking vehicle may leave the scene after a collision (hit-and-run "
    "or scrape-and-leave). Do not assume the first vehicle you see near a "
    "stopped car is the one that struck it — trace the timeline backwards.\n"
    "- Do NOT confuse vehicles that arrive AFTER a collision with the vehicle "
    "that CAUSED it. Temporal ordering is critical: establish which interaction "
    "happened first.\n"
    "- Still frames are provided at key timestamps alongside the video. Use "
    "them to verify spatial relationships between vehicles at critical moments."
)

print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# ── Task prompt — full scene description schema ────────────────────
TASK_PROMPT = """\
Analyse the attached video and the supplementary still frames, then
provide a complete scene description.

CRITICAL ANALYSIS INSTRUCTIONS:
- When a vehicle is observed stopped in an unusual position (e.g. stopped
  in a live lane on a roundabout), you MUST trace backwards in time to
  determine WHY it stopped. Look at the frames immediately before it became
  stationary to identify what caused it to halt.
- Pay very close attention to large vehicles (trucks, HGVs, lorries)
  navigating roundabouts and tight curves — their swept path often extends
  into adjacent lanes, causing sideswipe collisions with vehicles alongside
  them. A truck and car entering a roundabout side-by-side is a common
  collision scenario.
- Do NOT assume a vehicle was "inexplicably stopped" — there is almost
  always a cause. Look for interactions with other vehicles in the seconds
  immediately before the vehicle became stationary.
- When two vehicles are in close proximity on a roundabout (especially a
  large truck and a smaller car), examine whether the larger vehicle's
  turning arc encroached on the smaller vehicle's lane.
- A hit-and-run or scrape-and-leave scenario is possible: the striking
  vehicle may continue driving after the collision without stopping.
- Distinguish carefully between a vehicle that is STOPPED BECAUSE IT WAS
  HIT versus a vehicle that simply HAPPENS TO BE NEARBY a stopped vehicle.
  Temporal sequence matters: which event came first?
- Vehicles arriving at the scene AFTER a collision may simply be navigating
  around the already-stopped vehicle — do NOT misattribute the collision to
  a later-arriving vehicle unless you can clearly see an impact between them.

Return a JSON object with the following schema:

{
  "environment": {
    "weather": "<clear | rain | fog | snow | overcast | ...>",
    "lighting": "<daylight | dusk | dawn | night | artificial | ...>",
    "road_type": "<motorway | urban | suburban | rural | roundabout | intersection | ...>",
    "road_surface": "<dry | wet | icy | gravel | ...>",
    "traffic_density": "<empty | light | moderate | heavy | congested>",
    "additional_notes": "<any other relevant environmental observations>"
  },
  "detected_objects": [
    {
      "object_type": "<car | truck | bus | motorcycle | bicycle | pedestrian | traffic_sign | traffic_light | lane_marking | road_barrier | building | tree | animal | debris | other>",
      "description": "<brief description including colour, size, behaviour, or state>",
      "first_seen_seconds": <float or null>,
      "last_seen_seconds": <float or null>,
      "position_in_frame": "<centre | left_lane | right_lane | left_shoulder | right_shoulder | far_background | ...>",
      "confidence": <float 0-1>
    }
  ],
  "events": [
    {
      "event_type": "<collision | near_miss | hard_braking | lane_change | pedestrian_crossing | traffic_light_change | vehicle_overtake | vehicle_turning | emergency_vehicle | road_obstruction | other>",
      "timestamp_seconds": <float>,
      "timestamp_display": "<HH:MM:SS>",
      "description": "<what happened>",
      "involved_objects": ["<object_type>", ...],
      "severity": "<low | medium | high | null>",
      "confidence": <float 0-1>,
      "witnessed_crash_detail": {
        "impact_type": "<rear_end | head_on | side_swipe | t_bone | rollover | run_off_road | pedestrian_strike | cyclist_strike | multi_vehicle_pileup | none_near_miss | other>",
        "involved_parties": [
          {
            "road_user_type": "<car | truck | bus | motorcycle | bicycle | pedestrian | other>",
            "description": "<colour, make/model if visible, lane, direction>",
            "action_before_event": "<what they were doing just before>",
            "action_after_event": "<what they did after>",
            "apparent_damage": "<visible damage or 'none visible'>"
          }
        ],
        "probable_cause": "<primary cause assessment>",
        "fault_indicators": ["<observable cue suggesting fault>", ...],
        "sequence_of_events": "<step-by-step narrative of how the crash unfolded>",
        "post_crash_actions": "<what happened after: vehicles stopping, emergency response, debris, etc.>",
        "estimated_speed_range": "<low (< 20 mph) | moderate (20-40 mph) | high (> 40 mph) | null>",
        "dashcam_vehicle_involved": <true | false>
      }
    }
  ],
  "summary": "<overall narrative summary of everything observed in the footage>"
}

Instructions:
- List EVERY distinct object you can identify, not just vehicles.
- Include static infrastructure: signs, traffic lights, barriers, road markings.
- Include environmental features: buildings, trees, weather cues.
- For events, capture ALL notable actions — not just collisions.
  Examples: a car changing lanes, a pedestrian crossing, a traffic light
  turning red, a vehicle braking hard, an overtake, a turn at a junction.
- CRASH / NEAR-MISS EVENTS require special attention:
  * Set event_type to 'collision' or 'near_miss' as appropriate.
  * ALWAYS populate the 'witnessed_crash_detail' object for these events.
  * This applies whether the dashcam vehicle is directly involved in the
    crash OR is merely a bystander witnessing a crash between other vehicles.
  * Describe each involved party separately with their actions before and
    after the event, visible damage, and any other observable details.
  * Assess probable cause and list all fault indicators you can observe.
  * Provide a step-by-step sequence-of-events narrative.
  * Note any post-crash actions: vehicles stopping, hazard lights, emergency
    services, debris on road, other vehicles slowing down.
  * Set dashcam_vehicle_involved to true only if the recording vehicle
    itself was part of the collision; false if it was a witness.
  * If the striking vehicle leaves the scene, note this explicitly.
- For all NON-crash events, set 'witnessed_crash_detail' to null.
- Provide timestamps relative to the start of the video.
- Be conservative with confidence scores.
- If the same object persists across many seconds, report it once with
  first_seen_seconds and last_seen_seconds.
- Return ONLY the JSON object, no markdown fences or commentary.
"""

print("Task prompt length:  ", len(TASK_PROMPT), "chars")

In [ ]:
# ── Assemble the full messages payload ────────────────────────────
user_content = video_content_parts + [
    {"type": "text", "text": TASK_PROMPT}
]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]

print(f"Messages payload assembled — {len(messages)} messages, "
      f"{len(user_content)} user content parts.")

In [ ]:
# ── Call Gemini 2.5 Pro via LiteLLM (Pass 1) ──────────────────────────────────
import time as _time

TEMPERATURE = 0.0

logger.info("Pass 1: Sending multimodal request to %s …", MODEL_NAME)

_start = _time.perf_counter()

raw_response_pass1 = await litellm.acompletion(
    model=MODEL_NAME,
    messages=messages,
    temperature=TEMPERATURE,
    response_format={"type": "json_object"},
    cache={"no-store": True},
)

_elapsed = _time.perf_counter() - _start
logger.info(
    "Pass 1 response received — model: %s (%.2f s)",
    getattr(raw_response_pass1, "model", "unknown"),
    _elapsed,
)

## 6. Parse the Pass 1 Response into Pydantic Models

In [ ]:
# ── Parse Pass 1 response ─────────────────────────────────────────
from litellm.types.utils import ModelResponse, Choices

def extract_json_from_response(raw_response) -> dict:
    """Extract and parse JSON from a LiteLLM ModelResponse."""
    if not isinstance(raw_response, ModelResponse):
        raise TypeError(
            f"Expected ModelResponse, got {type(raw_response).__name__}."
        )
    choice = raw_response.choices[0]
    if not isinstance(choice, Choices):
        raise TypeError(
            f"Expected Choices, got {type(choice).__name__}."
        )
    content = choice.message.content
    if not content:
        raise ValueError("LLM returned empty content.")

    clean = content.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1]
        if clean.endswith("```"):
            clean = clean[:-3].strip()

    return json.loads(clean)


parsed_pass1 = extract_json_from_response(raw_response_pass1)
print("Pass 1 JSON parsed successfully.")
print(f"Top-level keys: {list(parsed_pass1.keys())}")

# Quick summary of what Pass 1 found
p1_events = parsed_pass1.get("events", [])
p1_crashes = [e for e in p1_events if e.get("event_type") in ("collision", "near_miss")]
print(f"Pass 1 detected {len(p1_events)} event(s), {len(p1_crashes)} crash/near-miss.")
for c in p1_crashes:
    print(f"  → [{c['event_type']}] at {c.get('timestamp_display', '?')}: {c.get('description', '')[:120]}")

## 7. Verification Pass (Pass 2)

Pass 2 sends the initial analysis back to the model along with the video and key frames, asking it to specifically verify:

1. **Causality** — why each stopped vehicle actually stopped.
2. **Temporal ordering** — which interaction happened first.
3. **Truck swept-path** — whether the Yodel truck's turning arc caused contact with adjacent vehicles.
4. **Later arrivals** — whether vehicles entering after the incident are merely navigating around the scene, not colliding.


In [ ]:
# ── Build the verification prompt ─────────────────────────────────
VERIFICATION_PROMPT = f"""\
You previously analysed this video and produced the following scene analysis:

{json.dumps(parsed_pass1, indent=2)}

Now re-examine the video AND the still frames with particular attention to
the following verification checks:

1. CAUSALITY CHECK: For every vehicle observed to be stationary in an
   unusual position, verify your explanation of WHY it stopped. Look at the
   frames 2-5 seconds BEFORE it became stationary. What was it doing? What
   other vehicles were near it at that moment?

2. TEMPORAL ORDERING: Verify the sequence of events. Which vehicles
   interacted with each other FIRST? If Vehicle A is already stopped when
   Vehicle B arrives, then B did not cause A to stop.

3. TRUCK SWEPT-PATH CHECK: Carefully examine the still frames at 15-22
   seconds. A large truck (green Yodel box truck) and a black SUV both enter
   the roundabout at approximately the same time from adjacent lanes.
   - Does the truck's turning path cause it to encroach on the black SUV's
     lane as both navigate the curve?
   - Could the black SUV stopping on the roundabout be a CONSEQUENCE of
     being struck or scraped by the truck, rather than a pre-existing
     condition?
   - Did the truck continue driving after any contact (hit-and-run)?

4. LATER ARRIVALS: Vehicles entering the roundabout AFTER the black SUV
   has already stopped (e.g. a red SUV arriving around 38-40s) may simply
   be navigating around the already-stopped vehicle. Verify whether any
   actual contact occurred between these later vehicles and the stopped SUV,
   or whether they are just passing by.

Based on your re-examination, provide a CORRECTED JSON output using the
exact same schema as before. Fix any temporal attribution errors,
misidentified involved parties, or incorrect causality chains. If your
original analysis was correct, return it unchanged.

Return ONLY the JSON object, no markdown fences or commentary.
"""

print(f"Verification prompt length: {len(VERIFICATION_PROMPT)} chars")

In [ ]:
# ── Assemble Pass 2 messages (reuse video + key frames) ───────────
verification_content = video_content_parts + [
    {"type": "text", "text": VERIFICATION_PROMPT}
]

messages_pass2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": verification_content},
]

print(f"Pass 2 messages assembled — {len(verification_content)} content parts.")

In [ ]:
# ── Call Gemini 2.5 Pro (Pass 2 — verification) ───────────────────
logger.info("Pass 2: Sending verification request to %s …", MODEL_NAME)

_start = _time.perf_counter()

raw_response_pass2 = await litellm.acompletion(
    model=MODEL_NAME,
    messages=messages_pass2,
    temperature=TEMPERATURE,
    response_format={"type": "json_object"},
    cache={"no-store": True},
)

_elapsed = _time.perf_counter() - _start
logger.info(
    "Pass 2 response received — model: %s (%.2f s)",
    getattr(raw_response_pass2, "model", "unknown"),
    _elapsed,
)

In [ ]:
# ── Parse Pass 2 response ─────────────────────────────────────────
parsed_pass2 = extract_json_from_response(raw_response_pass2)
print("Pass 2 JSON parsed successfully.")
print(f"Top-level keys: {list(parsed_pass2.keys())}")

# Use the verified output as the final result
parsed = parsed_pass2

# Quick comparison
p2_events = parsed.get("events", [])
p2_crashes = [e for e in p2_events if e.get("event_type") in ("collision", "near_miss")]
print(f"\nPass 2 detected {len(p2_events)} event(s), {len(p2_crashes)} crash/near-miss.")
for c in p2_crashes:
    print(f"  → [{c['event_type']}] at {c.get('timestamp_display', '?')}: {c.get('description', '')[:120]}")

# Flag differences
if len(p1_crashes) != len(p2_crashes):
    print(f"\n Pass 2 changed the number of crash events: {len(p1_crashes)} → {len(p2_crashes)}")
else:
    for i, (c1, c2) in enumerate(zip(p1_crashes, p2_crashes)):
        if c1.get("description") != c2.get("description"):
            print(f"\n Crash event {i+1} description changed between passes.")

## 8. Parse the Final Response into Pydantic Models

In [ ]:
# ── Build the structured output from the verified (Pass 2) JSON ───
environment = (
    EnvironmentContext(**parsed["environment"])
    if "environment" in parsed and parsed["environment"]
    else None
)

detected_objects = [
    DetectedObject(**obj) for obj in parsed.get("detected_objects", [])
]

# Parse events — handle nested witnessed_crash_detail for collisions/near-misses
events: list[SceneEvent] = []
for evt_raw in parsed.get("events", []):
    # Work on a copy to avoid mutating raw_output
    evt_data = dict(evt_raw)
    crash_detail_raw = evt_data.pop("witnessed_crash_detail", None)
    crash_detail = None
    if crash_detail_raw is not None and isinstance(crash_detail_raw, dict):
        detail_data = dict(crash_detail_raw)
        parties_raw = detail_data.pop("involved_parties", [])
        parties = [CrashInvolvedParty(**p) for p in parties_raw]
        crash_detail = WitnessedCrashDetail(involved_parties=parties, **detail_data)
    events.append(SceneEvent(witnessed_crash_detail=crash_detail, **evt_data))

analysis_output = VideoSceneAnalysisOutput(
    task_name="video_scene_analysis_demo",
    camera_type=CAMERA_TYPE,
    video_file_name=VIDEO_PATH.name,
    environment=environment,
    detected_objects=detected_objects,
    events=events,
    summary=parsed.get("summary"),
    raw_output=parsed,
)

# Summarise
n_crashes = sum(1 for e in events if e.event_type in ('collision', 'near_miss'))
n_other = len(events) - n_crashes
print(f"\nDetected {len(analysis_output.detected_objects)} object(s).")
print(f"Detected {len(events)} event(s): {n_crashes} crash/near-miss, {n_other} other.")
print(f"Summary: {analysis_output.summary}")

## 9. Explore the Results

In [ ]:
# ── Environment context ────────────────────────────────────────────
if analysis_output.environment:
    env = analysis_output.environment
    print("=" * 60)
    print("  Environment Context")
    print("=" * 60)
    print(f"  Weather:         {env.weather}")
    print(f"  Lighting:        {env.lighting}")
    print(f"  Road type:       {env.road_type}")
    print(f"  Road surface:    {env.road_surface}")
    print(f"  Traffic density: {env.traffic_density}")
    if env.additional_notes:
        print(f"  Notes:           {env.additional_notes}")
else:
    print("No environment context returned.")

In [ ]:
# ── Detected objects ───────────────────────────────────────────────
import pandas as pd

if analysis_output.detected_objects:
    df_objects = pd.DataFrame([o.model_dump() for o in analysis_output.detected_objects])
    display(
        df_objects.style
        .format({"confidence": "{:.2%}"})
        .set_caption(f"Detected Objects ({len(df_objects)} total)")
    )
else:
    print("No objects detected.")

In [ ]:
# ── Object type distribution ───────────────────────────────────────
if analysis_output.detected_objects:
    type_counts = df_objects["object_type"].value_counts()
    print("\nObject type distribution:")
    for obj_type, count in type_counts.items():
        print(f"  {obj_type:<20s} {count}")

In [ ]:
# ── Events ─────────────────────────────────────────────────────────
if analysis_output.events:
    for i, event in enumerate(analysis_output.events, 1):
        print(f"\n{'=' * 60}")
        print(f"  Event {i}: {event.event_type}")
        print(f"{'=' * 60}")
        print(f"  Timestamp:   {event.timestamp_display} ({event.timestamp_seconds:.1f}s)")
        print(f"  Severity:    {event.severity or 'N/A'}")
        print(f"  Confidence:  {event.confidence:.2%}")
        print(f"  Involved:    {', '.join(event.involved_objects) if event.involved_objects else 'N/A'}")
        print(f"  Description: {event.description}")
else:
    print("No notable events detected.")

In [ ]:
# ── Events as a DataFrame ─────────────────────────────────────────
if analysis_output.events:
    df_events = pd.DataFrame([e.model_dump() for e in analysis_output.events])
    # Convert involved_objects list to a comma-separated string for display
    df_events["involved_objects"] = df_events["involved_objects"].apply(
        lambda x: ", ".join(x) if x else ""
    )
    # Drop the nested crash detail column from the summary table
    display_cols = [c for c in df_events.columns if c != "witnessed_crash_detail"]
    display(
        df_events[display_cols].style
        .format({"confidence": "{:.2%}"})
        .set_caption(f"Scene Events ({len(df_events)} total)")
    )

In [ ]:
# ── Witnessed crash / near-miss details ────────────────────────────
crash_events = [e for e in analysis_output.events if e.witnessed_crash_detail is not None]

if crash_events:
    for i, event in enumerate(crash_events, 1):
        cd = event.witnessed_crash_detail
        assert cd is not None
        print(f"\n{'#' * 60}")
        print(f"  WITNESSED CRASH/NEAR-MISS {i} — {event.event_type.upper()}")
        print(f"{'#' * 60}")
        print(f"  Timestamp:              {event.timestamp_display} ({event.timestamp_seconds:.1f}s)")
        print(f"  Severity:               {event.severity or 'N/A'}")
        print(f"  Confidence:             {event.confidence:.2%}")
        print(f"  Impact type:            {cd.impact_type or 'N/A'}")
        print(f"  Dashcam involved:       {'Yes' if cd.dashcam_vehicle_involved else 'No (witness only)'}")
        print(f"  Probable cause:         {cd.probable_cause or 'N/A'}")
        print(f"  Estimated speed range:  {cd.estimated_speed_range or 'N/A'}")
        if cd.fault_indicators:
            print(f"  Fault indicators:       {', '.join(cd.fault_indicators)}")
        print(f"\n  Sequence of events:")
        print(f"    {cd.sequence_of_events or 'N/A'}")
        print(f"\n  Post-crash actions:")
        print(f"    {cd.post_crash_actions or 'N/A'}")
        print(f"\n  Description: {event.description}")
        assert cd is not None
        if cd.involved_parties:
            print(f"\n  Involved Parties ({len(cd.involved_parties)}):")
            for j, party in enumerate(cd.involved_parties, 1):
                print(f"    ── Party {j}: {party.road_user_type} ──")
                print(f"       Description:    {party.description}")
                print(f"       Before event:   {party.action_before_event or 'N/A'}")
                print(f"       After event:    {party.action_after_event or 'N/A'}")
                print(f"       Apparent damage: {party.apparent_damage or 'N/A'}")
else:
    print("No collisions or near-misses witnessed in this footage.")

In [ ]:
# ── Raw JSON output for inspection ─────────────────────────────────
display(JSON(analysis_output.raw_output, root="raw_model_output"))

In [ ]:
# ── Token usage summary (both passes) ─────────────────────────────
for label, resp in [("Pass 1", raw_response_pass1), ("Pass 2", raw_response_pass2)]:
    usage = getattr(resp, "usage", None)
    if usage:
        print(f"{label} Token Usage")
        print(f"  Prompt tokens:     {getattr(usage, 'prompt_tokens', 'N/A')}")
        print(f"  Completion tokens: {getattr(usage, 'completion_tokens', 'N/A')}")
        print(f"  Total tokens:      {getattr(usage, 'total_tokens', 'N/A')}")
    else:
        print(f"{label}: Token usage not available.")
    print()

In [ ]:
# ── Save results to disk ───────────────────────────────────────────
output_path = Path("./scene_analysis_results_causality.json")
output_path.write_text(json.dumps(analysis_output.model_dump(), indent=2, default=str))
print(f"Results saved to {output_path.resolve()}")

# Also save the Pass 1 output for comparison
pass1_path = Path("./scene_analysis_results_pass1.json")
pass1_path.write_text(json.dumps(parsed_pass1, indent=2, default=str))
print(f"Pass 1 (pre-verification) saved to {pass1_path.resolve()}")

---

### Notes

- **Schema evolution**: The `DetectedObject` and `SceneEvent` schemas are
  designed to be extensible. Add new `object_type` or `event_type` values
  as the model surfaces them — the schema uses `str` rather than `Literal`
  enums to allow for open-vocabulary detection.
- **Witnessed crash detail**: When `event_type` is `collision` or `near_miss`,
  the `witnessed_crash_detail` field is populated with forensic-level metadata:
  impact type, each involved party's description and pre/post-event actions,
  fault indicators, sequence of events, post-crash actions, and whether the
  dashcam vehicle was involved or just a bystander witness.
- **Crash-specific filtering**: To extract only crash events downstream,
  filter `events` by `event_type in ('collision', 'near_miss')` and access
  the `witnessed_crash_detail` field.
- **Multiple cameras**: Extend `video_content_parts` with additional
  video entries (one text label + one `image_url` per camera).
- **Backward compatibility**: The `VideoSceneAnalysisOutput` schema is a
  superset of the original `VideoAnalysisOutput`. Existing crash-event
  fields map directly to the `events` list with `event_type='collision'`.
- **Enhanced system prompt**: added causal reasoning principles, truck swept-path awareness, hit-and-run detection, and temporal attribution guidance
- **Enhanced task prompt**: added a `CRITICAL ANALYSIS INSTRUCTIONS` section that forces the model to trace causality backwards ("why did this vehicle stop?") and warns against common errors like blaming a later-arriving vehicle for an earlier incident
- **Key frame extraction**: OpenCV is used to extract still images at 21 critical timestamps (approach, roundabout entry, post-incident). These are sent alongside the video to provide pixel-level detail at the moments when vehicles interact closely.
- **Two-pass verification**: Pass 1 generates the initial analysis; Pass 2 sends that analysis back with explicit verification questions targeting temporal attribution, causality, and truck swept-path patterns. The final output uses the verified Pass 2 result.